# Driver Drowsiness Detection

This notebook integrates the complete real-time driver drowsiness
detection system.

The system combines:

- MediaPipe Face Landmarker for facial landmark detection
- Eye landmark-based eye region extraction
- A fine-tuned CNN for eye-state classification
- Mouth Aspect Ratio (MAR) for yawn detection
- Temporal drowsiness logic
- Real-time visual feedback
- An alarm system

The final system processes webcam frames continuously and displays
the driver's eye state, prediction confidence, mouth activity,
drowsiness indicators, and final status.

## 1. Import Required Libraries

The detection and inference logic is implemented inside the
`src` modules.

This notebook acts as the integration layer and does not
reimplement those components.

In [1]:
import cv2
import time
import numpy as np
import os
import sys
from pathlib import Path

PATH = Path.cwd().parent
sys.path.append(str(PATH))


In [2]:
from src.face_detection import FaceDetector
from src.eye_detection import EyeDetector, calculate_ear
from src.eye_inference import EyeInference
from src.yawn_detection import calculate_mar, is_yawning
from src.drowsiness_logic import DrowsinessDetector
from src.video_stream import VideoStream
from src.alarm import Alarm
import time


    def detect(self, frame, timestamp_ms):
        """
        Detect facial landmarks for the current frame.

        Parameters
        ----------
        frame : numpy.ndarray
            OpenCV BGR frame.

        timestamp_ms : int
            Increasing timestamp in milliseconds.

        Returns
        -------
        FaceLandmarkerResult
            MediaPipe face landmark result.
        """

        # OpenCV BGR → RGB
        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Convert to MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Synchronous detection
        result = self.landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        return result



In [3]:
cap = cv2.VideoCapture(1)

face_detector = FaceDetector()

timestamp = 0

while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read frame.")
        break

    timestamp += 33

    result = face_detector.detect(
        frame,
        timestamp
    )

    if len(result.face_landmarks) > 0:

        cv2.putText(
            frame,
            "FACE DETECTED",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "NO FACE DETECTED",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    cv2.imshow(
        "Face Detection Test",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
face_detector.close()
cv2.destroyAllWindows()

## 2. Initialize Detection Components

Each component is responsible for its own internal configuration.

- `FaceDetector` initializes MediaPipe.
- `EyeInference` loads the trained MRL model.
- `EyeDetector` extracts the eye regions.
- `DrowsinessDetector` maintains temporal state.

No duplicate model loading or MediaPipe initialization is required
in this notebook.

In [8]:
face_detector = FaceDetector()
eye_detector = EyeDetector()
eye_inference = EyeInference()
drowsiness_detector = DrowsinessDetector()

Loading model...
Model loading and warmup successful


## 3. Initialize Webcam

The webcam provides the continuous video stream that will be
processed frame by frame.

## 4. Real-Time Detection Pipeline

For every video frame:

1. Submit the frame to the Face Landmarker.
2. Retrieve the latest facial landmark result.
3. Extract the left and right eye regions.
4. Classify both eyes using the trained CNN.
5. Calculate the Mouth Aspect Ratio (MAR).
6. Detect a possible yawn.
7. Pass eye and yawn states to the drowsiness logic.
8. Display the results.
9. Trigger the alarm when drowsiness is detected.

The Face Landmarker operates in `LIVE_STREAM` mode, therefore
face detection is asynchronous and the latest available result
is retrieved from the detector.

In [9]:
# Alarm is now modularly handled via src.alarm.Alarm (Cross-Platform)
from src.alarm import Alarm
alarm = Alarm(frequency=1000, duration_ms=400, interval_ms=200)
print(f"Cross-platform Alarm ready using backend: {alarm.backend}")


In [ ]:
# 1. Initialize Threaded Camera Stream (Zero-Latency Capture)
stream = VideoStream(src=1, width=640, height=480).start()
print("Threaded VideoStream started successfully!")

# 2. Initialize Components for this session
face_detector = FaceDetector()
eye_detector = EyeDetector()
eye_inference = EyeInference()  # High-speed TFLite with XNNPACK
drowsiness_detector = DrowsinessDetector()
alarm = Alarm(frequency=1000, duration_ms=400, interval_ms=200)  # Cross-platform

previous_status = "Awake"

# Performance and live FPS tracking
fps_start_time = time.time()
fps_counter = 0
current_fps = 0.0

print("Starting real-time detection at max FPS... Press 'q' to quit.")

try:
    while True:
        grabbed, frame = stream.read()
        if not grabbed or frame is None:
            time.sleep(0.005)
            continue

        # Calculate live FPS
        fps_counter += 1
        fps_elapsed = time.time() - fps_start_time
        if fps_elapsed >= 0.5:
            current_fps = fps_counter / fps_elapsed
            fps_counter = 0
            fps_start_time = time.time()

        # Monotonically increasing real timestamp in ms
        current_timestamp_ms = int(time.time() * 1000)

        # Face Detection (MediaPipe)
        result = face_detector.detect(frame, current_timestamp_ms)

        if result is None or len(result.face_landmarks) == 0:
            cv2.putText(
                frame,
                "NO FACE DETECTED",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )
            cv2.putText(frame, f"FPS: {current_fps:.1f}", (frame.shape[1] - 140, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.imshow("Driver Drowsiness Detection", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
            continue

        face_landmark = result.face_landmarks[0]
        frame_height, frame_width = frame.shape[:2]

        # 3. Geometric Eye Aspect Ratio (0.02ms calculation)
        left_ear, right_ear, avg_ear = calculate_ear(face_landmark, frame_width, frame_height)

        # Extract eye crop boxes
        eye_data = eye_detector.detect(frame, face_landmark)
        left_eye = eye_data["left_eye"]
        right_eye = eye_data["right_eye"]
        left_box = eye_data["left_box"]
        right_box = eye_data["right_box"]

        # 4. Smart Hybrid Pre-Filtering
        # If eyes are clearly wide open (EAR > 0.28), instantly classify as Awake (0.02ms bypass)
        if avg_ear > 0.28:
            left_label, left_confidence, left_probability = "Awake", 0.99, 0.01
            right_label, right_confidence, right_probability = "Awake", 0.99, 0.01
        else:
            # Eyes are drooping or closed -> invoke fine-tuned TFLite CNN in a single forward pass
            (left_label, left_confidence, left_probability), (right_label, right_confidence, right_probability) = (
                eye_inference.predict_pair(left_eye, right_eye)
            )

        left_eye_closed = (left_label == "Sleepy")
        right_eye_closed = (right_label == "Sleepy")

        # 5. Mouth Aspect Ratio (Yawn Detection)
        mar = calculate_mar(face_landmark, frame_width, frame_height)
        yawning = is_yawning(mar)

        # 6. Drowsiness Temporal Logic State Machine
        drowsiness_result = drowsiness_detector.update(left_eye_closed, right_eye_closed, yawning)
        status = drowsiness_result["status"]

        # Draw Eye Bounding Boxes
        lx1, lx2, ly1, ly2 = left_box
        rx1, rx2, ry1, ry2 = right_box
        cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), (255, 255, 255), 2)
        cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), (255, 255, 255), 2)

        # HUD Telemetry
        cv2.putText(frame, f"FPS: {current_fps:.1f}", (frame.shape[1] - 140, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(frame, f"Left Eye: {left_label} {left_confidence * 100:.1f}%", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Right Eye: {right_label} {right_confidence * 100:.1f}%", (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"EAR: {avg_ear:.2f}", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"MAR: {mar:.2f}", (30, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Yawning: {'YES' if yawning else 'NO'}", (30, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Eye Closed: {drowsiness_result['eye_closed_duration']:.1f}s", (30, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Yawn Duration: {drowsiness_result['yawn_duration']:.1f}s", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        status_color = (0, 255, 0) if status == "Awake" else (0, 0, 255)
        cv2.putText(frame, f"STATUS: {status.upper()}", (30, 265), cv2.FONT_HERSHEY_SIMPLEX, 1.0, status_color, 3)
        # 7. Cross-Platform Alarm Trigger
        if status == "Drowsy":
            alarm.start()
        elif status == "Awake":
            alarm.stop()

        cv2.imshow("Driver Drowsiness Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    alarm.stop()
    stream.stop()
    face_detector.close()
    cv2.destroyAllWindows()
    print("Drowsiness detection system stopped.")


Loading model...
Model loading and warmup successful
Webcam opened successfully
Starting real-time detection... Press 'q' to quit.
Drowsiness detection system stopped.
